# DATASCI 151 - Introduction to Statistical Computing II
## Lecture 12 - SQLite in Python
**Author:** Danilo Freire (danilo.freire@emory.edu, Emory University)

# Hello! Great to see you again! 😊

# Lecture overview 📚

## Lecture overview

### Last time we learned: 

- What are relational databases and why they are still important in data science 
- How to create databases, tables, and insert data using basic SQL commands 
- How to perform queries with `SELECT`, `WHERE` and `ORDER BY` 
- Group data with `GROUP BY` and filter groups with `HAVING` 
- Statistics with `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` 
- That was a lot! 🤓 
- SQLite is generally easier to set up as it's often file-based or in-memory

### Today we will learn: 

- How to connect SQLite with Python
- We will use the built-in `sqlite3` library together with `pandas` 
- Other SQL commands, such as `LIKE`, `IN`, `BETWEEN`, `CASE`, window functions, and string functions like `SUBSTR` and `LENGTH` 
- How to fill missing data with `COALESCE` 
- How to use database cursors and fetch results (`fetchall`, `fetchone`) 
- Let's get started! 🚀

# Connecting SQLite with Python 🐍

## Connecting and Creating a Cursor

- To interact with an SQLite database from Python, we first need to establish a connection and then create a cursor.
- The **connection** links our script to the database file.
- The **cursor** is like a handle we use to send SQL commands and manage results.

In [1]:
import sqlite3

db_file = 'lecture12.db' # Database filename

# 1. Connect to the database (creates file if needed)
conn = sqlite3.connect(db_file) 

# 2. Create a cursor object to execute commands
cur = conn.cursor()

# Executing SQL in Python

## Creating Tables

- We define the structure of our data using `CREATE TABLE`.
- Use `cur.execute()` to run the command and `conn.commit()` to **save** the change.

In [2]:
# Use triple quotes for multi-line SQL
# Drop table if exist 
sql_drop = '''
DROP TABLE IF EXISTS drivers; 
'''

# Execute the SQL command
cur.execute(sql_drop)

# Create a new table named 'drivers'
sql_create = '''
CREATE TABLE drivers (
    driver_id INTEGER PRIMARY KEY AUTOINCREMENT,
    driver_name TEXT,
    team TEXT,
    nationality TEXT,
    victories INTEGER
);
'''
# Execute the SQL command
cur.execute(sql_create)

# Commit the transaction to save the table
conn.commit() 

# Check if the table was created
cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='drivers';")
print(cur.fetchall()) # Should show [('drivers',)] if successful

[('drivers',)]


## Inserting Data

- Once the table exists, we populate it using `INSERT INTO`.
- Again, `cursor.execute()` runs the command, and `conn.commit()` **makes the insertions permanent**.

In [3]:
# Insert a few rows
insert_query = '''
INSERT INTO drivers (driver_name, team, nationality, victories) VALUES
('Lewis Hamilton', 'Mercedes', 'British', 103),
('Max Verstappen', 'Red Bull Racing', 'Dutch', 55),
('Fernando Alonso', 'Aston Martin', NULL, NULL),
('Charles Leclerc', 'Ferrari', 'Monégasque', NULL);
'''

# Execute the insert command
cur.execute(insert_query)

# Commit the insertions
conn.commit()
print("Data inserted into drivers table.")

Data inserted into drivers table.


## Fetching All Results (`fetchall`) and One Result (`fetchone`)

- After a `SELECT` query, `fetchall()` retrieves **all results at once** into a list of tuples.
- Useful if you need the full dataset immediately, but can use more memory for large results.

In [4]:
cur.execute('SELECT driver_id, driver_name, team FROM drivers') 
all_rows = cur.fetchall() # Get the list of tuples

# Print the fetched data
print("All rows from drivers (driver_id, driver_name, team):")
for row in all_rows:
    print(row)

All rows from drivers (driver_id, driver_name, team):
(1, 'Lewis Hamilton', 'Mercedes')
(2, 'Max Verstappen', 'Red Bull Racing')
(3, 'Fernando Alonso', 'Aston Martin')
(4, 'Charles Leclerc', 'Ferrari')


- `fetchone()` retrieves rows **one by one**. Each call gets the next available row.
- Returns `None` when no more rows are left. Good for processing sequentially or just getting the top item.

In [5]:
cur.execute('SELECT driver_id, driver_name FROM drivers ORDER BY driver_id')

print("\nFetching rows one by one:")
row1 = cur.fetchone()
print("First row:", row1) 

row2 = cur.fetchone()
print("Second row:", row2)


Fetching rows one by one:
First row: (1, 'Lewis Hamilton')
Second row: (2, 'Max Verstappen')


## Iterating Over the Cursor

- A more "Pythonic" and **memory-efficient** way to process results row-by-row is to iterate directly over the cursor.

In [6]:
# Execute the query
query = "SELECT driver_name, team FROM drivers WHERE driver_name LIKE 'M%'"

print("\nDrivers starting with 'M' (iterating cursor):")
# Loop directly over the cursor object after execute
for row_tuple in cur.execute(query):
     print(row_tuple)


Drivers starting with 'M' (iterating cursor):
('Max Verstappen', 'Red Bull Racing')


- However, you can also use `.fetchall()` or `.fetchone()` after executing a query, depending on your needs.

In [7]:
# Fetch all results at once
print("\nDrivers starting with 'M' (using fetchall):")
results = cur.execute("SELECT driver_name, team FROM drivers WHERE driver_name LIKE 'M%'").fetchall()
for r in results:
    print(r)


Drivers starting with 'M' (using fetchall):
('Max Verstappen', 'Red Bull Racing')


# Filtering Data

## `IN` and `BETWEEN` Operators

- Use `IN` to check if a column's value matches **any value in a specified list**.

In [8]:
print("\nDrivers in Ferrari or Mercedes:")
results_in = cur.execute("SELECT driver_name, team FROM drivers WHERE team IN ('Ferrari', 'Mercedes')").fetchall()
for row in results_in:
    print(row)


Drivers in Ferrari or Mercedes:
('Lewis Hamilton', 'Mercedes')
('Charles Leclerc', 'Ferrari')


- `NOT IN` works similarly to exclude values in the list.

In [9]:
print("\nDrivers NOT in Ferrari or Mercedes:")
results_not_in = cur.execute("SELECT driver_name, team FROM drivers WHERE team NOT IN ('Ferrari', 'Mercedes')").fetchall()
for row in results_not_in:
    print(row)


Drivers NOT in Ferrari or Mercedes:
('Max Verstappen', 'Red Bull Racing')
('Fernando Alonso', 'Aston Martin')


- `BETWEEN` checks if a value is within a specified range. **Inclusive** of the endpoints.

In [10]:
print("\nDrivers with victories between 10 and 60:")
results_between = cur.execute("SELECT driver_name, victories FROM drivers WHERE victories BETWEEN 10 AND 60").fetchall()
for row in results_between:
    print(row)


Drivers with victories between 10 and 60:
('Max Verstappen', 55)


- `NOT BETWEEN` excludes values within the specified range.

## `LIKE` Operator: Basic Patterns

- `LIKE` enables simple pattern matching in text strings.
  - **`%`**: Matches any sequence (including none) of characters.
  - **`_`**: Matches exactly one character.

In [11]:
# Starts with 'L'
print("\nDrivers whose names start with 'L':")
cur.execute("SELECT driver_name FROM drivers WHERE driver_name LIKE 'L%'")
for row in cur.fetchall(): print(row)


Drivers whose names start with 'L':
('Lewis Hamilton',)


In [12]:
# Ends with 'Racing'
print("\nTeams ending with 'Racing':")
cur.execute("SELECT team FROM drivers WHERE team LIKE '%Racing'")
for row in cur.fetchall(): print(row)


Teams ending with 'Racing':
('Red Bull Racing',)


In [13]:
# 'e' as the second letter
print("\nDrivers with 'e' as the second letter:")
cur.execute("SELECT driver_name FROM drivers WHERE driver_name LIKE '_e%'")
for row in cur.fetchall(): print(row)


Drivers with 'e' as the second letter:
('Lewis Hamilton',)
('Fernando Alonso',)


## `LIKE` Operator: Case Sensitivity and `NOT LIKE`

- SQLite `LIKE` is case-insensitive by default. Use `LOWER()`/`UPPER()` for **reliable, explicit case handling**.

In [14]:
# Case-insensitive search for names starting with 'l'
print("\nDrivers starting with 'l' (using LOWER()):")
cur.execute("SELECT driver_name FROM drivers WHERE LOWER(driver_name) LIKE 'l%'")
for row in cur.fetchall(): print(row)


Drivers starting with 'l' (using LOWER()):
('Lewis Hamilton',)


- You can also use `COLLATE NOCASE` to make a specific comparison case-insensitive. **This is a more general (and recommended) approach.**

In [15]:
# Case-insensitive search for names starting with 'l' using COLLATE
print("\nDrivers starting with 'l' (using COLLATE NOCASE):")
cur.execute("SELECT driver_name FROM drivers WHERE driver_name LIKE 'l%' COLLATE NOCASE")
for row in cur.fetchall(): print(row)


Drivers starting with 'l' (using COLLATE NOCASE):
('Lewis Hamilton',)


- Use `NOT LIKE` to find strings that **do not match** the pattern.

## Try it yourself! 🤓 {#sec:exercise01}

- Practice time! Use `cur.execute()` and fetching/looping.
- Find all drivers whose names start with 'M'.
- Find drivers whose nationality has exactly 7 characters (**`LENGTH()`** function might be useful).
- List drivers whose names start with 'L' **or** 'M'.
- Find drivers with 1 to 10 victories (**`BETWEEN`**).

In [16]:
# Your SQL queries here, executed via cur.execute()


# Handling Missing Data

## Finding NULLs (`IS NULL`)  and `COALESCE`

- SQL represents missing values as `NULL`. Use `IS NULL` to find them. **`IS NOT NULL` finds rows with values**.

In [17]:
print("\nDrivers with NULL victories:")
cur.execute("SELECT driver_name, victories FROM drivers WHERE victories IS NULL")
for row in cur.fetchall(): print(row)


Drivers with NULL victories:
('Fernando Alonso', None)
('Charles Leclerc', None)


- `COALESCE(value1, value2, ...)` is useful for replacing `NULL`s with a default value. It returns the **first non-NULL expression** in its arguments.

In [18]:
# If 'victories' is NULL, display 0 instead.
print("\nVictories (filled NULLs with 0):")
cur.execute("SELECT driver_name, COALESCE(victories, 0) AS victories_filled FROM drivers")
for row in cur.fetchall(): print(row)


Victories (filled NULLs with 0):
('Lewis Hamilton', 103)
('Max Verstappen', 55)
('Fernando Alonso', 0)
('Charles Leclerc', 0)


## `COALESCE` with Subqueries

- The default value in `COALESCE` can be dynamic, calculated by a **subquery**.
- If `victories` is `NOT NULL`, it uses that value. Otherwise, it computes the average from non-NULL rows.
- The syntax is `COALESCE(column01, column02, ..., (subquery))`.
- `CAST` converts the result to a specific type, if needed.
- We use `CAST(... AS INTEGER)` to ensure the result is an integer. But this is optional.

In [19]:
query_coalesce_sub = '''
SELECT 
  driver_name,
  COALESCE(victories, 
    -- Subquery calculates average victories from non-NULL rows
    (SELECT CAST(AVG(victories) AS INTEGER) 
     FROM drivers 
     WHERE victories IS NOT NULL) 
  ) AS victories_imputed
FROM drivers; 
''' 
print("\nVictories (imputed NULLs with overall average):")
cur.execute(query_coalesce_sub)
for row in cur.fetchall(): print(row)


Victories (imputed NULLs with overall average):
('Lewis Hamilton', 103)
('Max Verstappen', 55)
('Fernando Alonso', 79)
('Charles Leclerc', 79)


- Here, `NULL` victories are replaced by the average of non-NULL victories.

# Window Functions

## Introduction to Window Functions

- Window functions compute values across a set of rows (a "window") related to the current row, without collapsing them like `GROUP BY`.
- Essential for ranking, running totals, moving averages, etc.
- Basic syntax involves the **`OVER()`** clause.
- **Requires SQLite version 3.25.0 or newer**.

In [20]:
# Add more data for better examples
more_drivers_data = [
    ('Valtteri Bottas', 'Mercedes', 'Finnish', 10),
    ('Sergio Perez', 'Red Bull Racing', 'Mexican', 5),
    ('Lando Norris', 'McLaren', 'British', 2),
    ('Esteban Ocon', 'Alpine', 'French', 1) 
]
# Check if data already exists to avoid duplicates 
cur.execute("SELECT COUNT(*) FROM drivers WHERE driver_name = 'Valtteri Bottas'")
if cur.fetchone()[0] == 0:
    cur.executemany('INSERT INTO drivers (driver_name, team, nationality, victories) VALUES (?, ?, ?, ?)', more_drivers_data)
    conn.commit()
    print(f"Added {len(more_drivers_data)} more drivers.")
else:
    print("Additional drivers already exist.")

print("\nFull drivers table after potential additions:")
cur.execute("SELECT * FROM drivers") 
for row in cur.fetchall(): print(row)

# Check SQLite Version
cur.execute("SELECT sqlite_version();")
print(f"\nSQLite Version: {cur.fetchone()[0]}")

Added 4 more drivers.

Full drivers table after potential additions:
(1, 'Lewis Hamilton', 'Mercedes', 'British', 103)
(2, 'Max Verstappen', 'Red Bull Racing', 'Dutch', 55)
(3, 'Fernando Alonso', 'Aston Martin', None, None)
(4, 'Charles Leclerc', 'Ferrari', 'Monégasque', None)
(5, 'Valtteri Bottas', 'Mercedes', 'Finnish', 10)
(6, 'Sergio Perez', 'Red Bull Racing', 'Mexican', 5)
(7, 'Lando Norris', 'McLaren', 'British', 2)
(8, 'Esteban Ocon', 'Alpine', 'French', 1)

SQLite Version: 3.51.0


## Window Functions: `AVG` and `RANK`

- Let's see common window functions in action:
- `AVG(...) OVER ()`: Average over the entire query result.
- `AVG(...) OVER (PARTITION BY col)`: Average within groups (partitions) defined by `col`.
- `RANK() OVER (ORDER BY col)`: Assigns rank based on `col` (**gaps possible for ties**).

In [21]:
query_window_avg_rank = '''
SELECT 
    driver_name, team, victories,
    ROUND(AVG(victories) OVER (), 2) AS avg_overall, 
    ROUND(AVG(victories) OVER (PARTITION BY team), 2) AS avg_team, 
    RANK() OVER (ORDER BY victories DESC) AS rank_overall 
FROM drivers
WHERE victories IS NOT NULL 
ORDER BY rank_overall; 
'''
print("\nWindow function results (AVG, RANK):")
cur.execute(query_window_avg_rank)
print([names[0] for names in cur.description]) # Print column names
for row in cur.fetchall(): print(row)


Window function results (AVG, RANK):
['driver_name', 'team', 'victories', 'avg_overall', 'avg_team', 'rank_overall']
('Lewis Hamilton', 'Mercedes', 103, 29.33, 56.5, 1)
('Max Verstappen', 'Red Bull Racing', 55, 29.33, 30.0, 2)
('Valtteri Bottas', 'Mercedes', 10, 29.33, 56.5, 3)
('Sergio Perez', 'Red Bull Racing', 5, 29.33, 30.0, 4)
('Lando Norris', 'McLaren', 2, 29.33, 2.0, 5)
('Esteban Ocon', 'Alpine', 1, 29.33, 1.0, 6)


## Window Functions vs. `GROUP BY`

- A key difference: `GROUP BY` **reduces** the number of rows to one per group. Window functions **maintain** all original rows and add new columns based on the window calculation.

In [22]:
# GROUP BY example (summarises)
cur.execute("SELECT team, ROUND(AVG(victories), 2) FROM drivers WHERE victories IS NOT NULL GROUP BY team")
print("--- GROUP BY Output (Collapsed) ---")
for row in cur.fetchall(): print(row)

# Window Function example (adds detail to each row)
cur.execute('''
    SELECT driver_name, team, victories, 
           ROUND(AVG(victories) OVER (PARTITION BY team), 2) as avg_in_team
    FROM drivers 
    WHERE victories IS NOT NULL ORDER BY team, victories DESC
''')
print("\n--- Window Function Output (All Rows) ---")
for row in cur.fetchall(): print(row)

--- GROUP BY Output (Collapsed) ---
('Alpine', 1.0)
('McLaren', 2.0)
('Mercedes', 56.5)
('Red Bull Racing', 30.0)

--- Window Function Output (All Rows) ---
('Esteban Ocon', 'Alpine', 1, 1.0)
('Lando Norris', 'McLaren', 2, 2.0)
('Lewis Hamilton', 'Mercedes', 103, 56.5)
('Valtteri Bottas', 'Mercedes', 10, 56.5)
('Max Verstappen', 'Red Bull Racing', 55, 30.0)
('Sergio Perez', 'Red Bull Racing', 5, 30.0)


## Try it yourself! 🤓 {#sec:exercise02}

- Use a window function:
- Select `driver_name`, `nationality`, `victories`.
- Add `rank_nationality`: rank drivers by `victories` **within** each `nationality`. (**`PARTITION BY`** needed).
- Exclude `NULL` victories.
- Order by `nationality`, then `rank_nationality`.
- Use `cur.execute()` and fetching/looping.

In [23]:
# Your SQL query here for Exercise 02


# String Manipulation

## Basic String Functions

- SQLite provides standard functions for text processing.
- `LENGTH(str)`, `UPPER(str)`, `LOWER(str)`.
- `SUBSTR(str, start, length)` extracts a portion of the string (**start is 1-indexed**).

In [24]:
query_str1 = '''
SELECT driver_name, 
    LENGTH(driver_name) AS len,
    UPPER(driver_name) AS upper,
    LOWER(driver_name) AS lower,
    SUBSTR(driver_name, 1, 4) AS first_four -- Get first 4 characters
FROM drivers LIMIT 4;
'''
print("\nString function examples (Part 1):")
cur.execute(query_str1)
for row in cur.fetchall(): print(row)


String function examples (Part 1):
('Lewis Hamilton', 14, 'LEWIS HAMILTON', 'lewis hamilton', 'Lewi')
('Max Verstappen', 14, 'MAX VERSTAPPEN', 'max verstappen', 'Max ')
('Fernando Alonso', 15, 'FERNANDO ALONSO', 'fernando alonso', 'Fern')
('Charles Leclerc', 15, 'CHARLES LECLERC', 'charles leclerc', 'Char')


## More String Functions

- `TRIM(str)` removes leading/trailing whitespace.
- `REPLACE(str, find, replace)` substitutes text.
- `INSTR(str, find)` finds the starting position of `find` within `str` (**0 if not found**).
- The **`||`** operator concatenates (joins) strings.

In [25]:
query_str2 = '''
SELECT driver_name, 
    REPLACE(driver_name, ' ', '_') AS replaced_space,
    INSTR(LOWER(driver_name), 'a') AS first_a_pos, 
    'Driver: ' || driver_name AS labelled_name -- Concatenation
FROM drivers LIMIT 4;
'''
print("\nString function examples (Part 2):")
cur.execute(query_str2)
for row in cur.fetchall(): print(row)


String function examples (Part 2):
('Lewis Hamilton', 'Lewis_Hamilton', 8, 'Driver: Lewis Hamilton')
('Max Verstappen', 'Max_Verstappen', 2, 'Driver: Max Verstappen')
('Fernando Alonso', 'Fernando_Alonso', 5, 'Driver: Fernando Alonso')
('Charles Leclerc', 'Charles_Leclerc', 3, 'Driver: Charles Leclerc')


# Conditional Logic: CASE

## Basic `CASE` Statement

- The `CASE` statement is SQL's way of implementing **if-then-else logic** within a query.
- `CASE WHEN condition THEN result ELSE default_result END`

In [26]:
query_case1 = '''
SELECT driver_name, victories,
    CASE 
        WHEN victories > 50 THEN 'Legend' 
        ELSE 'Great Driver (or N/A)'      
    END AS category 
FROM drivers;
'''
print("\nCASE statement example (Basic):")
cur.execute(query_case1)
for row in cur.fetchall(): print(row)


CASE statement example (Basic):
('Lewis Hamilton', 103, 'Legend')
('Max Verstappen', 55, 'Legend')
('Fernando Alonso', None, 'Great Driver (or N/A)')
('Charles Leclerc', None, 'Great Driver (or N/A)')
('Valtteri Bottas', 10, 'Great Driver (or N/A)')
('Sergio Perez', 5, 'Great Driver (or N/A)')
('Lando Norris', 2, 'Great Driver (or N/A)')
('Esteban Ocon', 1, 'Great Driver (or N/A)')


## `CASE` with Multiple Conditions

- You can chain multiple `WHEN` clauses. The **first condition that evaluates to true** determines the result for that row. The optional `ELSE` handles cases where no `WHEN` is true.

In [27]:
query_case2 = '''
SELECT driver_name, victories,
    CASE 
        WHEN victories > 100 THEN 'All-Time Great'
        WHEN victories > 50 THEN 'Legend'
        WHEN victories >= 10 THEN 'Race Winner'
        WHEN victories > 0 THEN 'Podium Potential'
        ELSE 'Data Missing or Zero Wins' 
    END AS status
FROM drivers ORDER BY victories DESC NULLS LAST;
'''
print("\nCASE statement example (Multiple Conditions):")
cur.execute(query_case2)
for row in cur.fetchall(): print(row)


CASE statement example (Multiple Conditions):
('Lewis Hamilton', 103, 'All-Time Great')
('Max Verstappen', 55, 'Legend')
('Valtteri Bottas', 10, 'Race Winner')
('Sergio Perez', 5, 'Podium Potential')
('Lando Norris', 2, 'Podium Potential')
('Esteban Ocon', 1, 'Podium Potential')
('Fernando Alonso', None, 'Data Missing or Zero Wins')
('Charles Leclerc', None, 'Data Missing or Zero Wins')


## `CASE` for Conditional NULL Handling

- `CASE` offers more nuanced control over handling `NULL`s compared to just `COALESCE`, allowing checks on **other columns**.

In [28]:
query_case_null = '''
SELECT driver_name, 
    -- Fill nationality based on name if NULL
    CASE 
        WHEN nationality IS NULL AND driver_name = 'Fernando Alonso' THEN 'Spanish'
        WHEN nationality IS NULL THEN 'Unknown' 
        ELSE nationality
    END AS nationality_filled,
    -- Fill victories based on team if NULL
    CASE 
        WHEN victories IS NULL AND team = 'Aston Martin' THEN 32 -- Educated guess! 
        WHEN victories IS NULL THEN 0 
        ELSE victories
    END AS victories_filled
FROM drivers WHERE driver_name LIKE 'F%' OR driver_name LIKE 'L%'; -- Limit output
'''
print("\nCASE statement for conditional NULL handling:")
cur.execute(query_case_null)
for row in cur.fetchall(): print(row)


CASE statement for conditional NULL handling:
('Lewis Hamilton', 'British', 103)
('Fernando Alonso', 'Spanish', 32)
('Lando Norris', 'British', 2)


## Try it yourself! 🤓 {#sec:exercise03}

- Use `CASE` to create `driver_level`:
  - 'Beginner' if victories < 10 (and not NULL)
  - 'Intermediate' if 10 <= victories <= 50
  - 'Expert' if victories > 50
  - 'Unknown' if victories is NULL (**Handle NULL first!**)
- Select `driver_name`, `victories`, `driver_level`.
- Use `cur.execute()` and fetching/looping.

In [29]:
# Your SQL query here for Exercise 03


# Using SQL with `pandas` 🐼

## Reading Data with `pandas.read_sql`

- Reading data using loops and `fetchall()` works, but `pandas` makes it much easier for analysis.
- **`pandas.read_sql()`** executes a `SELECT` query and loads the results directly into a DataFrame.

In [30]:
import pandas as pd # Import pandas
from IPython.display import display # For better display in Jupyter

# Pass the SQL query and the connection object
df = pd.read_sql('SELECT * FROM drivers', conn) 
print("DataFrame loaded from SQL:")
display(df) 

DataFrame loaded from SQL:


,driver_id,driver_name,team,nationality,victories
0,1,Lewis Hamilton,Mercedes,British,103.0
1,2,Max Verstappen,Red Bull Racing,Dutch,55.0
2,3,Fernando Alonso,Aston Martin,None,NaN
3,4,Charles Leclerc,Ferrari,Monégasque,NaN
4,5,Valtteri Bottas,Mercedes,Finnish,10.0
5,6,Sergio Perez,Red Bull Racing,Mexican,5.0
6,7,Lando Norris,McLaren,British,2.0
7,8,Esteban Ocon,Alpine,French,1.0


- This combines execution and fetching into one step, returning a structured table.

## `read_sql` with Any `SELECT` Query

- You can use complex `SELECT` statements, including joins, filtering, ordering, etc., within `read_sql`.

In [31]:
query_pd = """
SELECT driver_name, team, victories 
FROM drivers 
WHERE victories > 5 OR victories IS NULL
ORDER BY team
"""
df_filtered_ordered = pd.read_sql(query_pd, conn)
print("\nFiltered and ordered DataFrame from SQL:")
display(df_filtered_ordered)


Filtered and ordered DataFrame from SQL:


,driver_name,team,victories
0,Fernando Alonso,Aston Martin,NaN
1,Charles Leclerc,Ferrari,NaN
2,Lewis Hamilton,Mercedes,103.0
3,Valtteri Bottas,Mercedes,10.0
4,Max Verstappen,Red Bull Racing,55.0


- `pandas` handles retrieving all the data specified by your SQL query.

## Manipulating DataFrames from SQL

- Once loaded, the DataFrame behaves like any other `pandas` DataFrame. Apply all your data manipulation skills!

In [32]:
# df was loaded previously
if 'df' in locals() and not df.empty:
    avg_vic_pd = df.groupby('team')['victories'].mean().dropna()
    print("\nAverage victories by team (Pandas):")
    print(avg_vic_pd)
else:
    print("DataFrame 'df' not available.")


Average victories by team (Pandas):
team
Alpine              1.0
McLaren             2.0
Mercedes           56.5
Red Bull Racing    30.0
Name: victories, dtype: float64


In [33]:
# Example using pandas .query() method for filtering
if 'df' in locals() and not df.empty:
    print("\nPandas query on DataFrame:")
    display(df.query('victories > 50 and nationality == "British"'))
else:
    print("DataFrame 'df' not available.")


Pandas query on DataFrame:


,driver_id,driver_name,team,nationality,victories
0,1,Lewis Hamilton,Mercedes,British,103.0


## Writing DataFrames to SQL (`to_sql`)

- The reverse is also easy: `DataFrame.to_sql()` writes the DataFrame's contents into a database table.

In [34]:
# Example: Create a DataFrame with British drivers
if 'df' in locals() and not df.empty and 'nationality' in df.columns:
    df_british = df[df['nationality'] == 'British'].copy()

    # Write to a new table named 'british_drivers'
    # index=False: Prevents pandas index from becoming a DB column (important!)
    df_british.to_sql('british_drivers', conn, if_exists='replace', index=False)
    print("\n'british_drivers' table created/replaced in DB.")

    # Verify by reading it back using pandas
    print("\nContents of 'british_drivers' table from DB:")
    display(pd.read_sql('SELECT * FROM british_drivers', conn))
elif 'df' in locals() and not df.empty:
    print("Column 'nationality' not found in DataFrame 'df'.")
else:
    print("DataFrame 'df' not available to create 'british_drivers'.")


'british_drivers' table created/replaced in DB.

Contents of 'british_drivers' table from DB:


,driver_id,driver_name,team,nationality,victories
0,1,Lewis Hamilton,Mercedes,British,103.0
1,7,Lando Norris,McLaren,British,2.0


## Try it yourself! 🤓 {#sec:exercise04}

- 1. Create `employees` table (`employee_id` INTEGER PRIMARY KEY AUTOINCREMENT, `employee_name` TEXT, `department` TEXT, `salary` INTEGER) using `cursor.execute()`.
- 2. Insert data for Alice (HR, 50k), Bob (IT, 60k), Charlie (HR, 70k), David (IT, 80k) using `cursor.executemany()` or multiple `cursor.execute()` calls. Commit the changes.
- 3. Read the full `employees` table into a pandas DataFrame called `df_employees` using **`pd.read_sql()`**. Print `df_employees`.
- 4. Using **`df_employees`**, calculate and print the average salary per department.

In [35]:
# Your code here for Exercise 04


# Conclusion 📚

## Conclusion

- `sqlite3` provides the basic tools to interact with SQLite from Python.
- Key steps: `connect()`, `cursor()`, `execute()`, **`commit()`**.
- Fetch results using `fetchone()`, `fetchall()`, or by iterating the cursor.
- Common SQL commands work as expected.
- Window functions require **modern SQLite versions**.
- Use `CASE` + `GROUP BY` for pivoting.
- **`pandas.read_sql()`** greatly simplifies loading data for analysis.
- `df.to_sql()` saves DataFrames back to the database.

![](figures/pandasql5.jpg)
Source: [Susan Ibach (2020)](https://susanibach.wordpress.com/2020/01/07/pandas-for-sql-lovers-select-from-table/)

# And that's all for today! 🚀

# Have a great day! 😊